# DataMate V5 — dự đoán Public
Bật GPU T4 và Internet cho bước cài đặt/biên dịch. Thêm ZIP source V5, Public Test chính thức và GGUF đã xác minh vào Kaggle Input. Chạy theo thứ tự; sửa đường dẫn nếu tự tìm thấy nhiều kết quả. Kết quả chỉ nộp sau khi kiểm tra log model và xác minh thành công.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
PROJECT = None  # Path đến thư mục gốc có kaggle_setup.py, hoặc ZIP source
PUBLIC = None   # Path đến thư mục split có questions.jsonl, hoặc ZIP Public
MODEL = None    # Path đến Qwen3.5-9B-Q4_K_M.gguf; None để tự tìm/tải
SERVER = None   # Path llama-server tương thích; None để tự chuẩn bị

def one(paths, label):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise RuntimeError(f'Expected one {label}, found {paths}; set its path explicitly')
    return paths[0]
if PROJECT is None:
    dirs = [p.parent for p in INPUT.rglob('kaggle_setup.py') if (p.parent/'datamate').is_dir()]
    PROJECT = one(dirs or [p for p in INPUT.rglob('*.zip') if p.stem.casefold().startswith('datamate_v5')], 'project')
PROJECT = Path(PROJECT)
if PUBLIC is None:
    dirs = [p.parent for p in INPUT.rglob('questions.jsonl') if not (p.parent/'answers.jsonl').exists() and (p.parent/'rulebook.md').exists() and (p.parent/'data').is_dir()]
    PUBLIC = one(dirs or INPUT.rglob('public_test*.zip'), 'Public split')
PUBLIC = Path(PUBLIC)
if MODEL is None:
    models = list(INPUT.rglob('Qwen3.5-9B-Q4_K_M.gguf'))
    MODEL = one(models, 'model') if models else None
ROOT = WORK/'datamate_v5'
if PROJECT.is_dir():
    shutil.copytree(PROJECT, ROOT, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(PROJECT) as z: z.extractall(WORK)
assert (ROOT/'requirements.txt').is_file() and PUBLIC.exists()
sys.path.insert(0, str(ROOT))
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True)
run('-m', 'pip', 'install', '-r', 'requirements.txt')
print('PROJECT:', PROJECT, 'PUBLIC:', PUBLIC, 'MODEL:', MODEL)


In [ ]:
# Prepare runtime; validates the GGUF checksum.
runtime_file = WORK/'datamate_runtime_paths.json'
args = ['kaggle_setup.py', '--prepare', runtime_file]
if MODEL: args += ['--model', MODEL]
if SERVER: args += ['--server', Path(SERVER)]
run(*args)
runtime = json.loads(runtime_file.read_text())
policy = json.loads((ROOT/'confidence_policy.json').read_text())
assert policy['threshold'] is None, policy
print('Runtime:', runtime, 'threshold:', policy['threshold'])


In [ ]:
# Run once. Use a new output directory if preserving an earlier run.
OUT = WORK/'datamate_public_result'
assert not OUT.exists(), f'{OUT} already exists; choose a new output directory'
run('-m', 'datamate', 'predict', '--data', PUBLIC, '--output', OUT,
    '--work', WORK/'datamate_public_cache', '--model-file', runtime['model'],
    '--server-bin', runtime['server'], '--threshold-config', ROOT/'confidence_policy.json')


In [ ]:
# Validate all 1,000 unique Public IDs, BTC schema, and matching logs.
from datamate.data import Dataset
with Dataset(PUBLIC) as dataset:
    questions = dataset.jsonl('questions.jsonl')
assert len(questions) == 1000 and len({q['id'] for q in questions}) == 1000
run('verify_outputs.py', '--data', PUBLIC, '--output', OUT)
from collections import Counter
rows = [json.loads(line) for line in (OUT/'submission.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
logs = [json.loads(line) for line in (OUT/'logs'/'questions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
print('Answer types:', dict(Counter(row['type'] for row in rows)))
print('Runtime errors:', sum(bool(row.get('runtime_error')) for row in logs), 'loader errors:', sum(bool(row.get('loader_error')) for row in logs))
print('Inspect logs/llama-server.log and questions.jsonl for model failures before submitting.')
print('Submission:', OUT/'submission.jsonl', 'Logs:', OUT/'logs'/'questions.jsonl')
print('Download:', shutil.make_archive(str(WORK/'datamate_public_result'), 'zip', OUT))
